# **Modelni moslashtirish**

## **Avval shug'ullantirilgan modelni o'z ma'lumotimizga moslashtirish**

In [ ]:
!pip install -U datasets fsspec

In [ ]:
!pip install -U transformers

# **1. Ma’lumotlarni tayyorlash**

### ag_news ma’lumotlar to'plamini yuklang. Uning train qismidan 2000 ta, test qismidan esa 400 ta yangilikni ajratib oling.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("ag_news")
small_train = dataset["train"].shuffle(seed=42).select(list(range(2000)))
small_test = dataset['test'].shuffle(seed=42).select(list(range(400)))

# **2. Matnni tokenlash**

### bert-base-uncased modelining tokenayzeridan foydalanib, matnlarni 256 ta so‘z bilan cheklaydigan tokenlash funksiyasini yarating. Yaratilgan funksiyani train va test to‘plamlariga qo‘llang.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, padding='max_length', max_length=256)

tokenized_train = small_train.map(preprocess_function, batched=True)
tokenized_test = small_test.map(preprocess_function, batched=True)

# **3. Modelni qayta o'qitish (Fine-tuning)**

### bert-base-uncased modelini 4 ta label (yangilik turi) uchun moslashtiring. Modelni 1 epoch davomida o‘qiting. TrainingArguments va Trainer klasslaridan foydalanib, o‘qitish jarayonini amalga oshiring.

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=4)

In [ ]:
# from transformers import TrainingArguments, Trainer

# training_args = TrainingArguments(
#     output_dir="./results",
#     num_train_epochs=1,
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=16,
#     logging_steps=10,
#     report_to='none'
#     )

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=tokenized_train,
#     eval_dataset=tokenized_test
# )

# trainer.train()

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_steps=10,
    report_to='none'
    )

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test
)

trainer.train()

results = trainer.evaluate()
print(results)

# **4. Bashorat qilish funksiyasi va modelni saqlash**

### Kiritilgan matnni qaysi yangilik turiga (World, Sports, Business, Sci/Tech) tegishli ekanligini aniqlovchi predict_news_category funksiyasini yarating. So'ngra, o'qitilgan modelni torch.save yordamida news_model.pth nomi bilan saqlang.

In [ ]:
texts = [
    "The world has many problems in recent years.",
    "Football is a very popular and interesting game in the world.",
    "Many types of business development are happening nowadays by businessmen.",
    "We should give so much attention to education."
]

In [ ]:
import torch
import torch.nn.functional as F

In [ ]:
def predict_news_category(text_input):
    inputs = tokenizer(text_input, padding=True, truncation=True, return_tensors="pt", max_length=256)
    # Move inputs to GPU if available
    if torch.cuda.is_available():
        inputs = {k: v.to('cuda') for k, v in inputs.items()}
        model.to('cuda') # Ensure model is also on GPU
    else:
        model.to('cpu') # Ensure model is on CPU if no GPU

    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=1)
        predictions = torch.argmax(probs, dim=1)

    label_map = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}

    # If single text input, return single prediction
    if isinstance(text_input, str):
        pred_label = label_map[predictions.item()]
        pred_prob = probs[0, predictions.item()].item()
        return pred_label, pred_prob
    # If list of texts, return list of predictions
    else:
        results = []
        for i in range(len(text_input)):
            pred_label = label_map[predictions[i].item()]
            pred_prob = probs[i, predictions[i].item()].item()
            results.append((pred_label, pred_prob))
        return results

# Example usage with the defined texts
for text in texts:
    category, probability = predict_news_category(text)
    print(f"Text: '{text}' -> Category: {category}, Probability: {probability:.4f}")

In [ ]:
torch.save(model.state_dict(), 'news_model.pth')

In [ ]:
model.load_state_dict(torch.load('news_model.pth'))
model.eval()

# **5. Gradio veb-interfeysini yaratish**

### predict_news_category funksiyasi asosida ishlaydigan Gradio interfeysini yarating. Unga "Yangilik turini aniqlash" sarlavhasini va "Bu yerda yangilik matnini kiritib, uning turini bilib oling" degan tavsif bering. Yaratilgan interfeysni ishga tushiring.

## **DEPLOYMENT**

### Website yaratish va modeldan hayotda foydalanish

In [ ]:
import torch
import torch.nn.functional as F

def predict_news_category(text):
  inputs = tokenizer(text, padding=True, truncation=True, return_tensors="pt", max_length=256)

  # Move inputs and model to GPU if available, otherwise use CPU
  if torch.cuda.is_available():
      inputs = {k: v.to('cuda') for k, v in inputs.items()}
      model.to('cuda')
  else:
      model.to('cpu')

  with torch.no_grad():
    outputs = model(**inputs)
    probs = F.softmax(outputs.logits, dim=1)
    predictions = torch.argmax(probs, dim=1).item()

  label_map = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}
  return label_map[predictions]

In [ ]:
import gradio as gr

interface = gr.Interface(
    fn=predict_news_category,
    inputs="text",
    outputs="text",
    title="Yangilik turini aniqlash",
    description="Bu yerda yangilik matnini kiritib, uning turini bilib oling, faqat inglizcha matn kriting"
)

interface.launch(share=True)

## Bu model yangilik matnlarini tasniflash (classification) uchun yaratilgan. Uning asosiy vazifasi — berilgan inglizcha matnni tahlil qilib, uni to'rtta asosiy kategoriya: World (Jahon), Sports (Sport), Business (Biznes), Sci/Tech (Fan/Texnologiya) dan biriga tegishli ekanligini bashorat qilishdir. Model bert-base-uncased asosida o'qitilgan va Gradio veb-interfeysi orqali foydalanuvchilarga matn kiritish va natijani olish imkoniyatini beradi.
